# KPI Bar Charts (Section A4)

This notebook runs the halftime simulation and shows two vertical bar charts:
1. Average wait-time KPIs
2. People and service-use counts

### When you change config.yaml
After changing halftime settings (for example toilet capacity or seat_leave_rate), rerun these cells in order:
- Cell 2 (run simulation from config)
- Cell 3 (prepare chart values)
- Cell 4 (render updated charts)

In [1]:
# Cell 2 purpose: Import modules and run the simulation once from config.yaml.
import sys
from pathlib import Path
from IPython.display import HTML, display

sys.path.insert(0, str(Path('../src').resolve()))

from simulated_city.config import load_config
from simulated_city.simulation_core import simulate_halftime_from_app_config

app_config = load_config()
result = simulate_halftime_from_app_config(app_config)

print('Simulation complete.')
print(f'Average wait overall: {result.average_wait_s:.2f}s')
print(f'Missed kickoff: {result.missed_kickoff_count}')

Simulation complete.
Average wait overall: 118.70s
Missed kickoff: 437


In [2]:
# Cell 3 purpose: Prepare values for the two requested chart groups.
avg_time_values = {
    'Overall wait': result.average_wait_s,
    'Toilet wait': result.average_wait_toilet_s,
    'Cafe wait': result.average_wait_cafe_s,
    'Urinal wait': result.average_wait_urinal_s,
}

count_values = {
    'Stayed seated': result.stayed_seated_count,
    'Went down': result.went_down_count,
    'Down & back': result.went_down_made_back_count,
    'Made kickoff': result.made_kickoff_count,
    'Missed kickoff': result.missed_kickoff_count,
    'Women toilet use': result.women_toilet_served_count,
    'Men toilet use': result.men_toilet_served_count,
    'Men urinal use': result.men_urinal_served_count,
    'Total served tasks': result.total_served_tasks,
}

In [10]:
# Cell 4 purpose: Render two vertical bar charts side-by-side using HTML/CSS.
def _chart_html(
    title: str,
    values: dict[str, float],
    color: str,
    y_label: str,
    y_max: float | None = None,
    y_tick_step: int = 100,
) -> str:
    auto_max = max(values.values()) if values else 1
    if auto_max <= 0:
        auto_max = 1

    scale_max = y_max if (y_max is not None and y_max > 0) else auto_max

    bars = []
    for label, value in values.items():
        clamped_value = max(0.0, min(float(value), float(scale_max)))
        height_pct = (clamped_value / scale_max) * 100
        value_text = f"{value:.2f}" if isinstance(value, float) else str(value)
        bars.append(
            f"""
            <div class='bar-wrap'>
              <div class='bar-value'>{value_text}</div>
              <div class='bar' style='height:{height_pct:.1f}%; background:{color};'></div>
              <div class='bar-label'>{label}</div>
            </div>
            """
        )

    tick_lines = []
    if y_tick_step > 0:
        tick_value = 0
        while tick_value <= int(scale_max):
            tick_pct = (tick_value / scale_max) * 100
            tick_lines.append(
                f"""
                <div class='tick-line' style='bottom:{tick_pct:.1f}%;'>
                  <span class='tick-label'>{tick_value}</span>
                </div>
                """
            )
            tick_value += y_tick_step
        if int(scale_max) % y_tick_step != 0:
            tick_lines.append(
                f"""
                <div class='tick-line' style='bottom:100%;'>
                  <span class='tick-label'>{int(scale_max)}</span>
                </div>
                """
            )

    bars_html = ''.join(bars)
    ticks_html = ''.join(tick_lines)
    y_text = f"{y_label} (0-{int(scale_max)})"
    return f"""
    <div class='panel'>
      <h3>{title}</h3>
      <div class='ylabel'>{y_text}</div>
      <div class='plot-shell'>
        <div class='y-grid'>{ticks_html}</div>
        <div class='plot-area'>{bars_html}</div>
      </div>
    </div>
    """

avg_panel = _chart_html('Average Time KPIs', avg_time_values, '#3b82f6', 'Seconds', y_max=400, y_tick_step=100)
count_panel = _chart_html('People & Service Use KPIs', count_values, '#10b981', 'Count', y_max=1000, y_tick_step=100)

page = f"""
<style>
  .kpi-grid {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 24px;
    align-items: start;
    margin-top: 12px;
  }}
  .panel {{
    border: 1px solid #d1d5db;
    border-radius: 10px;
    padding: 12px 12px 8px 12px;
    background: #ffffff;
  }}
  .panel h3 {{
    margin: 0 0 4px 0;
    font-size: 16px;
  }}
  .ylabel {{
    font-size: 12px;
    color: #6b7280;
    margin-bottom: 8px;
  }}
  .plot-shell {{
    position: relative;
    margin-left: 30px;
  }}
  .plot-area {{
    height: 340px;
    display: flex;
    align-items: stretch;
    gap: 8px;
    border-left: 1px solid #9ca3af;
    border-bottom: 1px solid #9ca3af;
    padding: 8px 8px 0 8px;
    overflow-x: auto;
    position: relative;
    z-index: 2;
    background: transparent;
  }}
  .y-grid {{
    position: absolute;
    inset: 8px 8px 0 8px;
    left: 0;
    right: 0;
    bottom: 0;
    pointer-events: none;
    z-index: 1;
  }}
  .tick-line {{
    position: absolute;
    left: 1px;
    right: 0;
    border-top: 1px dashed #e5e7eb;
  }}
  .tick-label {{
    position: absolute;
    left: -34px;
    top: -8px;
    font-size: 10px;
    color: #6b7280;
  }}
  .bar-wrap {{
    min-width: 84px;
    height: 100%;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: flex-end;
  }}
  .bar-value {{
    font-size: 11px;
    margin-bottom: 4px;
    color: #111827;
  }}
  .bar {{
    width: 48px;
    border-radius: 6px 6px 0 0;
    min-height: 2px;
  }}
  .bar-label {{
    margin-top: 6px;
    font-size: 11px;
    text-align: center;
    color: #111827;
    line-height: 1.2;
  }}
</style>
<div class='kpi-grid'>
  {avg_panel}
  {count_panel}
</div>
"""

display(HTML(page))